# 02: Normalize and upsert into Silver

Keep the latest update for each record ID across the Bronze history. Delta `MERGE` updates matching rows and inserts new IDs, so reruns leave one row per ID in Silver.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

BRONZE_TABLE = 'demo_bronze_activity'
SILVER_TABLE = 'demo_silver_activity'
required = {'record_id', 'activity_date', 'amount', 'updated_at'}

source = spark.table(BRONZE_TABLE)
missing = required - set(source.columns)
if missing:
    raise ValueError(f'Missing source fields: {sorted(missing)}')

# Trim IDs, cast dates and amounts, and reject records we can't safely merge.
cleaned = (source
    .select(
        F.trim('record_id').alias('record_id'),
        F.to_date('activity_date').alias('activity_date'),
        F.col('amount').cast('double').alias('amount'),
        F.to_timestamp('updated_at').alias('updated_at'),
        F.col('source_file'),
        F.col('ingested_at'),
    ))
invalid = cleaned.filter(
    F.col('record_id').isNull() | (F.col('record_id') == '') |
    F.col('activity_date').isNull() | F.col('amount').isNull() |
    F.col('updated_at').isNull()
)
if invalid.limit(1).count():
    raise ValueError(f'Invalid source rows: {invalid.count()} (fix the sample reports first)')

# Most recent source update wins. File name breaks ties deterministically.
window = Window.partitionBy('record_id').orderBy(
    F.col('updated_at').desc(), F.col('source_file').desc(), F.col('ingested_at').desc()
)
latest = (cleaned.withColumn('rank', F.row_number().over(window))
    .filter(F.col('rank') == 1).drop('rank'))

if not spark.catalog.tableExists(SILVER_TABLE):
    latest.write.format('delta').mode('overwrite').saveAsTable(SILVER_TABLE)
else:
    target = DeltaTable.forName(spark, SILVER_TABLE)
    # Do not overwrite a newer target record with an older source update.
    (target.alias('t').merge(latest.alias('s'), 't.record_id = s.record_id')
        .whenMatchedUpdateAll(condition='s.updated_at >= t.updated_at')
        .whenNotMatchedInsertAll()
        .execute())

print(f'Silver now contains {spark.table(SILVER_TABLE).count()} unique records')
